In [55]:
import pandas as pd
import ast
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast
import torch
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification
from torch.utils.data import DataLoader
from transformers import AdamW
from tqdm import tqdm
import numpy as np





In [56]:
df = pd.read_csv("ARP_PreLabel.csv")

def extract_entity_names(entities_str):
    try:
        entities = ast.literal_eval(entities_str)
        return [e[0] for e in entities if isinstance(e, tuple)]
    except:
        return []

df["Entity_Texts"] = df["Entities"].apply(extract_entity_names)


In [57]:
ENTITY_LIST = [
    "Federal Reserve", "Interest Rates", "Inflation", "Employment", "Unemployment", "GDP", "Trade", "Congress", "Monetary Policy", "Financial Stability", 
    "Price Stability", "Regulatory Implementation", "Pandemic", "Asset Runoff", "Reinvestment", "Money Market", "Bond Market", "Equity Markets", "Financial Markets", "Repo Markets", 
    "Fiscal Policy", "Balance Sheet", "Reserves", "Digital Dollar", "Foreign Currencies", "Federal Funds", "Demand", "Securities", "War", "Finance", 
    "Debt", "Mortgage", "Maturity", "Credit", "Labor Market", "Auction", "Press Conference", "Banking System", "Uncertain", "Development", "Economic Outlook", "Countries"
]


In [58]:
# label vector (0 or 1) for each sentence
def label_vector_from_entities(entity_names):
    vec = [0] * len(ENTITY_LIST)
    for i, ent in enumerate(ENTITY_LIST):
        if ent in entity_names:
            vec[i] = 1
    return vec

df["Label_Vector"] = df["Entity_Texts"].apply(label_vector_from_entities)


In [59]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)

In [60]:
# converted to strings
train_df["Sentence"] = train_df["Sentence"].astype(str)
test_df["Sentence"]  = test_df["Sentence"].astype(str)

In [61]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")

# encode
train_encodings = tokenizer(train_df["Sentence"].tolist(), truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(test_df["Sentence"].tolist(),  truncation=True, padding=True, max_length=128)

train_labels = train_df["Label_Vector"].tolist()
test_labels  = test_df["Label_Vector"].tolist()


In [62]:

class MultiLabelDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        return {
            key: torch.tensor(val[idx]) for key, val in self.encodings.items()
        } | {
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

    def __len__(self):
        return len(self.labels)

train_dataset = MultiLabelDataset(train_encodings, train_labels)
test_dataset  = MultiLabelDataset(test_encodings, test_labels)


In [63]:

model = BertForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(ENTITY_LIST),
    problem_type="multi_label_classification"
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
# prepare training data & device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
optimizer = AdamW(model.parameters(), lr=2e-5)

# calculate pos_weight
label_matrix = np.array(train_labels)  # shape: [num_samples, 42]
pos_counts = label_matrix.sum(axis=0)
neg_counts = len(label_matrix) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float).to(device)

# initialise the weighted loss function
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

model.train()
for epoch in range(30):
    total_loss = 0
    print(f"Epoch {epoch+1}")
    
    for batch in tqdm(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    
    print(f"Avg loss: {total_loss / len(train_loader):.4f}")


/opt/anaconda3/envs/ARP/lib/python3.12/site-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1


100%|██████████| 65/65 [02:04<00:00,  1.91s/it]


Avg loss: 1.2958
Epoch 2


100%|██████████| 65/65 [01:59<00:00,  1.84s/it]


Avg loss: 1.1518
Epoch 3


100%|██████████| 65/65 [01:56<00:00,  1.80s/it]


Avg loss: 1.0321
Epoch 4


  9%|▉         | 6/65 [00:12<02:00,  2.03s/it]


KeyboardInterrupt: 

In [ ]:
#model.save_pretrained(f"MicroF1_0.85")
#tokenizer.save_pretrained(f"MicroF1_0.85")

In [ ]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np

model.eval()
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()  

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        all_preds.append(preds)
        all_labels.append(labels)


y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

In [ ]:
print("F1-Score Evaluation:")

print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_true, y_pred, average="weighted"))

report = classification_report(
    y_true, y_pred, target_names=ENTITY_LIST, zero_division=0
)
print(report)


In [ ]:
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np
from tqdm import tqdm


model_path = "MicroF1_0.85" 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

tokenizer = BertTokenizerFast.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()


all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()  

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        preds = (probs > 0.5).astype(int) 

        all_preds.append(preds)
        all_labels.append(labels)


y_pred = np.vstack(all_preds)
y_true = np.vstack(all_labels)

print("Micro F1:", f1_score(y_true, y_pred, average="micro"))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Precision (micro):", precision_score(y_true, y_pred, average="micro"))
print("Recall (micro):", recall_score(y_true, y_pred, average="micro"))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=ENTITY_LIST))